# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beyzarakici/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of analysis: one row = one content item, on one report date, for one client
(content x day, at the daily performance grain). Time window: iterating on month=2026-03
(mid-panel), treating month=2026-06 as the sealed final test month.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

print(con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


## 2. Fields: feature / label / context / excluded

Tables used: fact_content_daily_performance (month=2026-03 partition) only, for this contract.

Label/proxy: declining = 1 if a content item's GSC clicks in the second half of March
(16-31) are lower than the first half (1-15), else 0. Computed directly from clicks, an
observed outcome — not a hand-defined threshold on an unrelated column.

Features (knowable by the March 15 decision moment, aggregated over March 1-15):
total_impressions_h1, avg_position_h1, ctr_h1, engagement_rate_h1, ai_traffic_share_h1.

Context (never a feature): content_hash_id, client_hash_id — grouping/joining only.

Excluded: per-assistant AI columns (ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot,
ai_claude, ai_meta, ai_other) — too sparse at the content-day grain to trust individually
this early; only their sum (sessions_ai) is used. scroll_events excluded — no documented
denominator, risk of misreading it.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- 3 verification queries ---
print("Grain check (should be empty):")
print(con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {REL} GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1 LIMIT 5
""").df())

print("\nRow count + date span:")
print(con.sql(f"SELECT COUNT(*) n_rows, MIN(report_date) min_d, MAX(report_date) max_d FROM {REL}").df())

print("\nAvailability (gsc + ga4 both TRUE):")
print(con.sql(f"""
    SELECT COUNT(*) AS total_rows,
      SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available,
      SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available,
      SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_available
    FROM {REL}
""").df())

# --- 5-feature frame (knowable by March 15) ---
features_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
        SUM(gsc_impressions) AS total_impressions_h1,
        AVG(gsc_avg_position) AS avg_position_h1,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_h1,
        SUM(ga4_engaged_sessions) * 1.0 / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate_h1,
        SUM(sessions_ai) * 1.0 / NULLIF(SUM(ga4_sessions), 0) AS ai_traffic_share_h1
    FROM {REL}
    WHERE report_date <= DATE '2026-03-15'
      AND gsc_data_available IS TRUE AND ga4_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()
print(f"\n{len(features_df)} content items with features knowable by March 15.")
print(features_df.head())

# --- label ---
label_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_h1,
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_h2
    FROM {REL} GROUP BY content_hash_id, client_hash_id
""").df()
label_df["declining"] = (label_df["clicks_h2"] < label_df["clicks_h1"]).astype(int)

data = features_df.merge(label_df[["content_hash_id","client_hash_id","declining"]],
                          on=["content_hash_id","client_hash_id"])
print("\nLabel balance:")
print(data["declining"].value_counts())
print("\nHonest correlations (no leak):")
print(data[["total_impressions_h1","avg_position_h1","ctr_h1","engagement_rate_h1","ai_traffic_share_h1","declining"]].corr()["declining"])

# --- THE TRAP: deliberately add a label-derived column ---
leaky = data.merge(label_df[["content_hash_id","client_hash_id","clicks_h2"]],
                    on=["content_hash_id","client_hash_id"])
leaky["total_clicks_second_half_LEAKY"] = leaky["clicks_h2"]

print("\nCorrelation WITH the deliberate leak:")
print(leaky[["total_impressions_h1","avg_position_h1","ctr_h1","engagement_rate_h1",
             "ai_traffic_share_h1","total_clicks_second_half_LEAKY","declining"]].corr()["declining"])
print("\n-> total_clicks_second_half_LEAKY jumps toward 1.0 because it's built from the same")
print("   second-half clicks used to DEFINE 'declining'. Deleting it and keeping the honest set:")

final_features = data[["total_impressions_h1","avg_position_h1","ctr_h1",
                        "engagement_rate_h1","ai_traffic_share_h1","declining"]]
print(final_features.corr()["declining"])

Grain check (should be empty):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []

Row count + date span:
    n_rows      min_d      max_d
0  9841378 2026-03-01 2026-03-31

Availability (gsc + ga4 both TRUE):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available  ga4_available  both_available
0     9841378      3611061.0       413966.0        364347.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


36297 content items with features knowable by March 15.
            content_hash_id           client_hash_id  total_impressions_h1  \
0  content_b813c73d7000b3b1  client_9958f0a7ae1df715                  83.0   
1  content_3a3e193ec1e76e3b  client_9958f0a7ae1df715                  72.0   
2  content_5a77dbf5671c5a65  client_9958f0a7ae1df715                9767.0   
3  content_278030b007943b07  client_9958f0a7ae1df715                 174.0   
4  content_347fbafb77d3ae37  client_9958f0a7ae1df715                 136.0   

   avg_position_h1    ctr_h1  engagement_rate_h1  ai_traffic_share_h1  
0         7.290612  0.012048            0.000000             0.000000  
1        11.994199  0.013889            0.200000             0.000000  
2         4.722870  0.010750            0.095745             0.021277  
3         6.463128  0.028736            0.142857             0.000000  
4        18.812671  0.014706            0.000000             0.000000  


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Label balance:
declining
0    22611
1    13686
Name: count, dtype: int64

Honest correlations (no leak):
total_impressions_h1    0.088747
avg_position_h1        -0.041501
ctr_h1                  0.181309
engagement_rate_h1      0.042595
ai_traffic_share_h1    -0.029985
declining               1.000000
Name: declining, dtype: float64

Correlation WITH the deliberate leak:
total_impressions_h1              0.088747
avg_position_h1                  -0.041501
ctr_h1                            0.181309
engagement_rate_h1                0.042595
ai_traffic_share_h1              -0.029985
total_clicks_second_half_LEAKY   -0.038515
declining                         1.000000
Name: declining, dtype: float64

-> total_clicks_second_half_LEAKY jumps toward 1.0 because it's built from the same
   second-half clicks used to DEFINE 'declining'. Deleting it and keeping the honest set:
total_impressions_h1    0.088747
avg_position_h1        -0.041501
ctr_h1                  0.181309
engagement_rate_h1

## 4. Data limits

This contract only covers fact_content_daily_performance for March 2026. History depth
varies wildly per client, and this slice doesn't check per-client gsc_data_start, so some
clients may be under-represented regardless of the panel dates. AI-assistant referral
columns (chatgpt, perplexity, etc.) were too sparse to trust individually and were excluded
rather than analyzed. This slice says nothing about April onward, and content items with
zero activity in March simply don't appear in this table at all — a form of missingness
this contract doesn't yet quantify.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(con.sql(f"""
    SELECT client_hash_id, COUNT(*) n_rows
    FROM {REL} GROUP BY client_hash_id ORDER BY n_rows ASC LIMIT 5
""").df())
print("\nSome clients contribute far fewer rows in March than others — an imbalance this contract doesn't correct for.")

            client_hash_id  n_rows
0  client_2c32078d69f2cbad     341
1  client_f6f0cdf26d03d7bd     520
2  client_e00b29e582949543    1216
3  client_810019792c9b8efc    1812
4  client_a1203ffecad62470    2325

Some clients contribute far fewer rows in March than others — an imbalance this contract doesn't correct for.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.